# ASL Detection Model
## Feature Extractor

In [1]:
import os
import cv2
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = 'hand_landmarker.task' # This is the file you just downloaded
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)

data, labels = [], []
directories = ['C:/Users/aliaa/Train_Alphabet', 'C:/Users/aliaa/Test_Alphabet']

for root_dir in directories:
    if not os.path.exists(root_dir):
        print(f"Skipping {root_dir} - folder not found.")
        continue
    
    for label_name in os.listdir(root_dir):
        label_path = os.path.join(root_dir, label_name)
        if os.path.isdir(label_path):
            print(f"Processing letter: {label_name}...")
            
            for img_name in os.listdir(label_path)[:500]: 
                img = cv2.imread(os.path.join(label_path, img_name))
                if img is None: continue
                
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                result = detector.detect(mp_image)
                
                if result.hand_landmarks:
                    coords = []
                    hand = result.hand_landmarks[0]
                    
                    # FIX: Get the Wrist (Landmark 0) coordinates as the "Base"
                    base_x = hand[0].x
                    base_y = hand[0].y

                    for landmark in hand:
                        # FIX: Subtract the wrist position from every point
                        # This makes the data about the SHAPE, not the LOCATION
                        coords.append(landmark.x - base_x)
                        coords.append(landmark.y - base_y)
                        
                    data.append(coords)
                    labels.append(label_name)
                    
df = pd.DataFrame(data)
df['target'] = labels
df.to_csv('asl_landmarks.csv', index=False)
print("Done! Check your folder for 'asl_landmarks.csv'.")

Processing letter: A...
Processing letter: B...
Processing letter: Blank...
Processing letter: C...
Processing letter: D...
Processing letter: E...
Processing letter: F...
Processing letter: G...
Processing letter: H...
Processing letter: I...
Processing letter: J...
Processing letter: K...
Processing letter: L...
Processing letter: M...
Processing letter: N...
Processing letter: O...
Processing letter: P...
Processing letter: Q...
Processing letter: R...
Processing letter: S...
Processing letter: T...
Processing letter: U...
Processing letter: V...
Processing letter: W...
Processing letter: X...
Processing letter: Y...
Processing letter: Z...
Processing letter: A...
Processing letter: B...
Processing letter: Blank...
Processing letter: C...
Processing letter: D...
Processing letter: E...
Processing letter: F...
Processing letter: G...
Processing letter: H...
Processing letter: I...
Processing letter: J...
Processing letter: K...
Processing letter: L...
Processing letter: M...
Processi

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle

df = pd.read_csv('asl_landmarks.csv')

X = df.drop('target', axis=1) # The 42 coordinates
y = df['target']              # The letters (A, B, C...)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

y_predict = model.predict(X_test)
score = accuracy_score(y_test, y_predict)
print(f"Model Accuracy: {score * 100:.2f}%")

with open('asl_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved as 'asl_model.pkl'!")

Model Accuracy: 98.10%
Model saved as 'asl_model.pkl'!
